# Tái hiện pipeline WHU-LX XGB-DQN

Notebook này được viết lại từ ý tưởng của file `scripts/reproduce_whulx_pipeline.py`, nhưng **không còn phụ thuộc vào file script đó**. Toàn bộ hàm xử lý dữ liệu, huấn luyện XGBoost, xây dựng DQN, rollout và đánh giá đều được đặt trực tiếp trong notebook.

Mục tiêu của notebook:

- Tái hiện pipeline XGB-DQN của repo WHU-LX ở mức có thể chạy lại được.
- Giải thích rõ từng bước để dùng trong báo cáo và thuyết trình.
- Dùng dữ liệu `data/Cleaned_data.csv` đã được đưa vào repo, nên người khác clone repo có thể chạy notebook mà không cần thư mục `../references`.
- Lưu kết quả ra `artifacts/outputs/whulx_reproduction/` để báo cáo DOCX có thể cập nhật lại từ file kết quả.

## 1. Chuẩn bị thư viện và đường dẫn

Pipeline cần các nhóm thư viện chính:

- `pandas`, `numpy`: đọc và xử lý dữ liệu dạng bảng.
- `scikit-learn`: chia train/test, mã hóa nhãn và tính metric.
- `xgboost`: huấn luyện mô hình dự đoán biến thiên nhiệt độ trong nhà.
- `tensorflow`: xây dựng và huấn luyện mạng DQN.

Dữ liệu được đọc trực tiếp từ `data/Cleaned_data.csv`. Đây là bản sao dữ liệu từ nguồn WHU-LX đã được đưa vào repo để notebook tự chạy được sau khi clone.

In [ ]:
import json
import math
import random
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "artifacts" / "outputs" / "whulx_reproduction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "Cleaned_data.csv"

np.random.seed(2022)
random.seed(2022)
tf.random.set_seed(2022)

print("Project root:", PROJECT_ROOT)
print("Dữ liệu:", DATA_PATH)
print("Thư mục lưu kết quả:", OUT_DIR)

## 2. Định nghĩa vùng tiện nghi nhiệt ASHRAE

Reward của DQN cần biết nhiệt độ trong nhà có đang nằm trong vùng tiện nghi hay không. Repo WHU-LX dùng vùng tiện nghi thích nghi theo ASHRAE.

Ý tưởng chính:

- Khi nhiệt độ ngoài trời thấp hơn hoặc bằng ngưỡng dưới, dùng dải tiện nghi thấp.
- Khi nhiệt độ ngoài trời cao hơn hoặc bằng ngưỡng trên, dùng dải tiện nghi cao.
- Nếu nhiệt độ ngoài trời nằm giữa hai ngưỡng, nội suy tuyến tính để lấy dải tiện nghi.

Trong notebook này, hàm `comfort_bounds()` trả về bốn mốc của dải tiện nghi; hàm `comfort_ok()` kiểm tra nhiệt độ trong nhà có nằm trong khoảng tiện nghi chính hay không.

In [ ]:
STANDARD_BANDS = {
    "ASHRAE": [[17.4, 18.4, 23.4, 24.4], [23.6, 24.6, 29.6, 30.6], [10, 30]]
}


def comfort_bounds(outdoor_temp, standard="ASHRAE"):
    l11, l12, u12, u11 = STANDARD_BANDS[standard][0]
    l21, l22, u22, u21 = STANDARD_BANDS[standard][1]
    t1, t2 = STANDARD_BANDS[standard][2]

    if outdoor_temp <= t1:
        return l11, l12, u12, u11
    if outdoor_temp >= t2:
        return l21, l22, u22, u21

    increase_l = (outdoor_temp - t1) * (l21 - l11) / (t2 - t1)
    increase_u = (outdoor_temp - t1) * (u21 - u11) / (t2 - t1)
    return l11 + increase_l, l12 + increase_l, u12 + increase_u, u11 + increase_u


def comfort_ok(indoor_temp, outdoor_temp):
    _, lower, upper, _ = comfort_bounds(float(outdoor_temp))
    return lower <= float(indoor_temp) <= upper

## 3. Thiết kế reward và action space

DQN cần hai thành phần quan trọng: action và reward.

### Action space

Pipeline gốc dùng 24 action:

- `0`: tắt điều hòa, đóng cửa sổ.
- `1-11`: bật điều hòa, đóng cửa sổ, đặt nhiệt độ mục tiêu từ 20 đến 30 độ C.
- `12`: tắt điều hòa, mở cửa sổ.
- `13-23`: bật điều hòa và mở cửa sổ, đặt nhiệt độ mục tiêu từ 20 đến 30 độ C.

### Reward

Reward gồm hai phần:

- Phạt sai lệch tiện nghi: nếu nhiệt độ trong nhà nằm ngoài vùng tiện nghi thì phạt theo bình phương khoảng cách.
- Phạt năng lượng: action có bật điều hòa sẽ bị trừ chi phí; action vừa bật điều hòa vừa mở cửa sổ bị phạt nặng hơn.

Vì chi phí điều hòa trong reward khá lớn, agent có xu hướng ưu tiên các action không bật điều hòa nếu vẫn giữ được comfort tương đối ổn.

In [ ]:
def calculate_reward(state, action, next_state):
    indoor_temp = float(next_state[0])
    outdoor_temp = float(next_state[2])
    _, lower, upper, _ = comfort_bounds(outdoor_temp)

    if lower <= indoor_temp <= upper:
        reward = 0.0
    elif indoor_temp < lower:
        reward = -((indoor_temp - lower) ** 2)
    else:
        reward = -((indoor_temp - upper) ** 2)

    if action in {0, 12}:
        reward += 0.0
    elif action > 12:
        reward -= 2 * (60 * 0.87 * 1)
    else:
        reward -= 60 * 0.87 * 1
    return float(reward)


def map_action_to_dataframe(action):
    action = int(action)
    target_temp, ac_status, window_status, c_last_time, w_last_time = 0, 0, 0, 0, 0

    if action == 0:
        pass
    elif 0 < action < 12:
        target_temp = 19 + action
        ac_status = 1
        c_last_time = 60
    elif action == 12:
        window_status = 1
        w_last_time = 60
    else:
        target_temp = action + 7
        ac_status = 1
        window_status = 1
        c_last_time = 60
        w_last_time = 60

    return target_temp, ac_status, window_status, c_last_time, w_last_time

## 4. Đọc và tiền xử lý dữ liệu

Dữ liệu đầu vào là `Cleaned_data.csv` từ WHU-LX. Dù tên file là cleaned, pipeline vẫn làm thêm một số bước để đảm bảo mô hình học được:

- Đọc CSV với encoding `gbk` vì dữ liệu gốc dùng encoding này.
- Chuyển `Date_Time` sang kiểu thời gian.
- Loại bỏ missing value.
- Loại bỏ giá trị lỗi `-999`.
- Mã hóa các cột dạng chuỗi sang số bằng `LabelEncoder`.

Kết quả gồm `raw` là dữ liệu gốc và `data` là dữ liệu đã xử lý để huấn luyện.

In [ ]:
def load_and_prepare_data(data_path):
    raw = pd.read_csv(data_path, encoding="gbk")
    data = raw.copy()

    data["Date_Time"] = pd.to_datetime(data["Date_Time"])
    data = data.dropna()
    data = data[data != -999].dropna()

    for col in data.columns:
        if data[col].dtype == "object":
            encoder = LabelEncoder()
            data[col] = encoder.fit_transform(data[col])

    return raw, data


raw, data = load_and_prepare_data(DATA_PATH)

print("Kích thước dữ liệu gốc:", raw.shape)
print("Kích thước sau xử lý:", data.shape)
data.head()

## 5. Huấn luyện XGBoost để dự đoán chuyển trạng thái

Trong pipeline này, XGBoost không phải là mô hình điều khiển. Nó đóng vai trò như một **mô hình môi trường gần đúng**.

Cụ thể:

- Input của XGBoost là trạng thái hiện tại cộng với thông tin action/trạng thái điều khiển.
- Target là `Differ_Indoor_Temp`, tức độ thay đổi nhiệt độ trong nhà ở bước tiếp theo.
- Khi DQN chọn action, XGBoost dự đoán `Differ_Indoor_Temp` để cập nhật `Indoor_Temp` tiếp theo.

Các metric theo dõi:

- MAE: sai số tuyệt đối trung bình.
- RMSE: sai số bình phương trung bình lấy căn, nhạy hơn với lỗi lớn.
- R2: mức độ mô hình giải thích được biến thiên của target.

In [ ]:
def train_xgboost(data, device="cpu"):
    x_data = data.drop(
        ["Next_Indoor_Temp", "Next_Indoor_RH", "Date_Time", "Study_ID", "Differ_Indoor_Temp", "ID"],
        axis=1,
    )
    y_data = data["Differ_Indoor_Temp"]

    x_train, x_test, y_train, y_test = train_test_split(
        x_data, y_data, test_size=0.2, random_state=2022
    )

    model = xgb.XGBRegressor(
        random_state=2000,
        verbosity=0,
        n_jobs=-1,
        tree_method="hist",
        device=device,
        max_depth=5,
        learning_rate=0.23474,
        n_estimators=500,
    )
    model.fit(x_train, y_train)

    pred = model.predict(x_test)
    mse = mean_squared_error(y_test, pred)
    metrics = {
        "x_shape": list(x_data.shape),
        "train_shape": list(x_train.shape),
        "test_shape": list(x_test.shape),
        "mae": float(mean_absolute_error(y_test, pred)),
        "rmse": float(math.sqrt(mse)),
        "r2": float(r2_score(y_test, pred)),
    }
    return model, metrics


model_xgb, xgb_metrics = train_xgboost(data, device="cpu")
xgb_metrics

## 6. Chọn một ngày mẫu để rollout

Notebook gốc của WHU-LX kiểm tra DQN trên một ngày mẫu. Hàm `choose_day()` lấy một đoạn dữ liệu theo ngày và tạo hai bảng:

- `data_test`: chứa các biến state mà DQN quan sát.
- `xgboost_test`: chứa đầy đủ feature để XGBoost dự đoán chuyển trạng thái.

State của DQN lấy 8 biến đầu:

`Indoor_Temp`, `Indoor_RH`, `Outdoor_Temp`, `Outdoor_RH`, `Rain`, `Cloud`, `Windspeed`, `Hour`.

Khung điều khiển chính là từ step 6 đến step 17, tức 12 bước điều khiển trong ngày.

In [ ]:
def choose_day(start_index, data):
    data_test_00 = data.iloc[start_index : start_index + 23].copy()
    data_test_0a = data_test_00.reset_index(drop=True)
    data_test_0 = data_test_00.reset_index(drop=True)

    data_test_0.at[0, "CLast_Time_T"] = (
        data_test_0a.iloc[0]["CLast_Time_T"] - data_test_0a.iloc[0]["AC_Status"] * 60
    )
    data_test_0.at[0, "WLast_Time_T"] = (
        data_test_0a.iloc[0]["WLast_Time_T"] - data_test_0a.iloc[0]["Window_Status"] * 60
    )

    data_test = data_test_0[
        [
            "Indoor_Temp",
            "Indoor_RH",
            "Outdoor_Temp",
            "Outdoor_RH",
            "Rain",
            "Cloud",
            "Windspeed",
            "Hour",
            "Next_Outdoor_Temp",
            "Next_Outdoor_RH",
        ]
    ].copy()

    xgboost_test = data_test_0.drop(
        ["Next_Indoor_Temp", "Next_Indoor_RH", "Date_Time", "Study_ID", "Differ_Indoor_Temp", "ID"],
        axis=1,
    )
    return data_test, xgboost_test


DAY_INDEX = 0
data_test, xgboost_test = choose_day(DAY_INDEX * 24, data)

print("data_test:", data_test.shape)
print("xgboost_test:", xgboost_test.shape)
data_test.head()

## 7. Xây dựng Replay Buffer và mạng DQN

DQN là một mạng neural network nhận state và trả về Q-value cho từng action.

Kiến trúc dùng trong pipeline:

- Dense 64, ReLU.
- Dense 64, ReLU.
- Dense 24, linear output, mỗi node tương ứng một action.

Replay buffer lưu các transition `(state, action, next_state, reward)`. Khi đủ batch size, agent lấy ngẫu nhiên một batch để cập nhật Q-network. Cách này giúp giảm tương quan giữa các mẫu liên tiếp trong rollout.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, next_state, reward):
        self.buffer.append((state, action, next_state, reward))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)


class DQN(tf.keras.Model):
    def __init__(self, num_actions):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(64, activation="relu")
        self.dense2 = tf.keras.layers.Dense(64, activation="relu")
        self.output_layer = tf.keras.layers.Dense(num_actions, activation="linear")

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return self.output_layer(x)

## 8. Hàm cập nhật Q-network và chính sách epsilon-greedy

Ở mỗi lần cập nhật, target Q được tính theo công thức DQN cơ bản:

`target = reward + gamma * max_a Q(next_state, a)`

Sau đó mô hình chỉ so sánh target với Q-value của action đã thực sự chọn. Hàm `epsilon_greedy_policy()` dùng để cân bằng giữa khám phá và khai thác:

- Với xác suất `epsilon`, chọn action ngẫu nhiên.
- Ngược lại, chọn action có Q-value cao nhất.

In [ ]:
def update_q_network(q_network, replay_buffer, optimizer, loss_fn, gamma, num_actions, batch_size):
    states, actions, next_states, rewards = zip(*replay_buffer.sample(batch_size))
    states = tf.convert_to_tensor(np.array(states), dtype=tf.float32)
    next_states = tf.convert_to_tensor(np.array(next_states), dtype=tf.float32)
    actions = tf.convert_to_tensor(np.array(actions), dtype=tf.int32)
    rewards = tf.convert_to_tensor(np.array(rewards), dtype=tf.float32)

    with tf.GradientTape() as tape:
        q_values = q_network(states)
        target_q_values = q_network(next_states)
        target_q_values = rewards + gamma * tf.reduce_max(target_q_values, axis=1)
        mask = tf.one_hot(actions, num_actions)
        q_action = tf.reduce_sum(q_values * mask, axis=1)
        loss = loss_fn(target_q_values, q_action)

    grads = tape.gradient(loss, q_network.trainable_variables)
    optimizer.apply_gradients(zip(grads, q_network.trainable_variables))
    return float(loss.numpy())


def epsilon_greedy_policy(q_network, state, epsilon, num_actions):
    if np.random.rand() < epsilon:
        return int(np.random.randint(num_actions))
    q_values = q_network(np.array([state], dtype=np.float32))
    return int(np.argmax(q_values[0]))

## 9. Rollout một ngày bằng XGBoost + DQN

Hàm `rollout_day()` là lõi của pipeline.

Có hai giai đoạn:

1. **Warm-up từ step 0 đến 5**: đặt action bằng 0 để dùng XGBoost cập nhật trạng thái ban đầu trước khung điều khiển.
2. **Điều khiển từ step 6 đến 17**: DQN chọn action, action được đưa vào bảng feature, XGBoost dự đoán nhiệt độ kế tiếp, sau đó tính reward và cập nhật replay buffer nếu đang train.

Hàm này cũng hỗ trợ `fixed_action` để đánh giá các baseline cố định như luôn tắt AC, luôn mở cửa hoặc luôn bật AC ở một setpoint nhất định.

In [ ]:
def rollout_day(
    model_xgb,
    q_network,
    data_test,
    xgboost_test,
    epsilon,
    train=False,
    replay_buffer=None,
    train_cfg=None,
    fixed_action=None,
):
    data_pre_test = data_test.copy()
    xgboost_pre_test = xgboost_test.copy()
    num_features = 8
    actions = []
    rewards = []
    losses = []

    for step in range(6):
        xgboost_pre_test.loc[
            step,
            ["Target_Temp", "AC_Status", "Window_Status", "CLast_Time", "WLast_Time", "CLast_Time_T", "WLast_Time_T"],
        ] = 0
        hour_row_df = pd.DataFrame(xgboost_pre_test.iloc[step]).T
        next_differ_temp = model_xgb.predict(hour_row_df)[0]
        next_in_temp = xgboost_pre_test.iloc[step]["Indoor_Temp"] + next_differ_temp
        xgboost_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp
        data_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp

    state = data_pre_test.iloc[6, :num_features].values

    for step in range(6, 18):
        if fixed_action is None:
            action = epsilon_greedy_policy(q_network, state, epsilon, train_cfg["num_actions"])
        else:
            action = int(fixed_action)

        target_temp, ac_status, window_status, c_last_time, w_last_time = map_action_to_dataframe(action)
        actions.append(action)

        xgboost_pre_test.at[step, "Target_Temp"] = target_temp
        xgboost_pre_test.at[step, "AC_Status"] = ac_status
        xgboost_pre_test.at[step, "Window_Status"] = window_status
        xgboost_pre_test.at[step, "CLast_Time"] = c_last_time
        xgboost_pre_test.at[step, "WLast_Time"] = w_last_time
        xgboost_pre_test.at[step, "CLast_Time_T"] = (
            xgboost_pre_test.iloc[step - 1]["CLast_Time_T"] + c_last_time if c_last_time > 0 else 0
        )
        xgboost_pre_test.at[step, "WLast_Time_T"] = (
            xgboost_pre_test.iloc[step - 1]["WLast_Time_T"] + w_last_time if w_last_time > 0 else 0
        )

        hour_row_df = pd.DataFrame(xgboost_pre_test.iloc[step]).T
        next_differ_temp = model_xgb.predict(hour_row_df)[0]
        next_in_temp = xgboost_pre_test.iloc[step]["Indoor_Temp"] + next_differ_temp
        xgboost_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp
        data_pre_test.at[step + 1, "Indoor_Temp"] = next_in_temp

        next_state = data_pre_test.iloc[step + 1, :num_features].values
        reward = calculate_reward(state, action, next_state)
        rewards.append(reward)

        if train and replay_buffer is not None:
            replay_buffer.push(state, action, next_state, reward)
            if len(replay_buffer) >= train_cfg["batch_size"]:
                losses.append(
                    update_q_network(
                        q_network,
                        replay_buffer,
                        train_cfg["optimizer"],
                        train_cfg["loss_fn"],
                        train_cfg["gamma"],
                        train_cfg["num_actions"],
                        train_cfg["batch_size"],
                    )
                )
        state = next_state

    return data_pre_test, actions, rewards, losses

## 10. Hàm tổng hợp metric

Các metric chính dùng để so sánh controller:

- `comfort_pct`: phần trăm timestep nằm trong vùng tiện nghi.
- `ac_on_pct`: phần trăm timestep bật điều hòa.
- `window_open_pct`: phần trăm timestep mở cửa sổ.
- `mean_indoor_temp`: nhiệt độ trong nhà trung bình.
- `total_reward`: tổng reward trong rollout.

Human baseline được lấy trực tiếp từ dữ liệu gốc, còn các controller khác được rollout lại bằng XGBoost.

In [ ]:
def summarize_day(data_day, actions):
    control = data_day.iloc[6:18].copy()
    comfort = [comfort_ok(row["Indoor_Temp"], row["Outdoor_Temp"]) for _, row in control.iterrows()]
    ac_on = sum(1 for a in actions if 0 < a < 12 or a > 12)
    window_open = sum(1 for a in actions if a >= 12)

    return {
        "comfort_pct": float(np.mean(comfort) * 100),
        "ac_on_pct": float(ac_on / len(actions) * 100) if actions else 0.0,
        "window_open_pct": float(window_open / len(actions) * 100) if actions else 0.0,
        "mean_indoor_temp": float(control["Indoor_Temp"].mean()),
    }


def summarize_human(data_test):
    control = data_test.iloc[6:18].copy()
    comfort = [comfort_ok(row["Indoor_Temp"], row["Outdoor_Temp"]) for _, row in control.iterrows()]
    return {
        "comfort_pct": float(np.mean(comfort) * 100),
        "mean_indoor_temp": float(control["Indoor_Temp"].mean()),
    }


def summarize_human_with_actions(data_test, xgboost_test):
    metrics = summarize_human(data_test)
    control = xgboost_test.iloc[6:18].copy()
    metrics["ac_on_pct"] = float(control["AC_Status"].mean() * 100)
    metrics["window_open_pct"] = float(control["Window_Status"].mean() * 100)
    return metrics


def aggregate_metric_rows(rows):
    if not rows:
        return {}
    keys = sorted({key for row in rows for key in row if isinstance(row.get(key), (int, float, np.number))})
    return {key: float(np.mean([row[key] for row in rows if key in row])) for key in keys}

## 11. Khởi tạo và huấn luyện DQN

Cấu hình huấn luyện bám theo pipeline gốc:

- `num_actions = 24`.
- `batch_size = 32`.
- `gamma = 0.9`.
- `epsilon` bắt đầu từ 1.0 và giảm dần đến tối thiểu 0.1.
- `EPISODES = 1000` để tái hiện cấu hình gốc.

Nếu chỉ muốn chạy thử nhanh, có thể giảm `EPISODES` xuống 10 hoặc 50. Tuy nhiên, kết quả dùng trong báo cáo nên giữ 1000 episodes.

In [ ]:
EPISODES = 1000
num_actions = 24

q_network = DQN(num_actions)
q_network(np.zeros((1, 8), dtype=np.float32))

replay_buffer = ReplayBuffer(10000)
optimizer = tf.optimizers.Adam(0.001)
loss_fn = tf.losses.MeanSquaredError()

epsilon = 1.0
min_epsilon = 0.1
epsilon_decay = 0.995

train_cfg = {
    "num_actions": num_actions,
    "batch_size": 32,
    "gamma": 0.9,
    "optimizer": optimizer,
    "loss_fn": loss_fn,
}

history = []

for episode in range(EPISODES):
    _, actions, rewards, losses = rollout_day(
        model_xgb,
        q_network,
        data_test,
        xgboost_test,
        epsilon,
        train=True,
        replay_buffer=replay_buffer,
        train_cfg=train_cfg,
    )

    if epsilon > min_epsilon:
        epsilon *= epsilon_decay

    history.append(
        {
            "episode": episode + 1,
            "reward": float(np.sum(rewards)),
            "epsilon": float(epsilon),
            "avg_loss": float(np.mean(losses)) if losses else np.nan,
        }
    )

    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}/{EPISODES}, reward={np.sum(rewards):.2f}, epsilon={epsilon:.3f}")

history_df = pd.DataFrame(history)
history_df.tail()

## 12. Đánh giá DQN trên một ngày mẫu

Sau khi huấn luyện, ta rollout lại ngày mẫu với `epsilon = 0.0`. Khi đó agent không chọn ngẫu nhiên nữa mà luôn chọn action có Q-value cao nhất.

Kết quả một ngày mẫu giúp kiểm tra nhanh pipeline có hoạt động đúng hay không. Tuy nhiên, không nên dùng một ngày duy nhất để kết luận mô hình tốt hay xấu trên toàn bộ dataset.

In [ ]:
eval_day, eval_actions, eval_rewards, _ = rollout_day(
    model_xgb,
    q_network,
    data_test,
    xgboost_test,
    epsilon=0.0,
    train=False,
    train_cfg={"num_actions": num_actions},
)

human_metrics = summarize_human(data_test)
dqn_metrics = summarize_day(eval_day, eval_actions)

action_counts = pd.Series(eval_actions).value_counts().sort_index()
action_distribution = {
    int(action): float(count / len(eval_actions) * 100)
    for action, count in action_counts.items()
}

single_day_summary = pd.DataFrame(
    [
        {"controller": "Human", **human_metrics},
        {
            "controller": "DQN",
            **dqn_metrics,
            "total_reward": float(np.sum(eval_rewards)),
            "actions": eval_actions,
            "action_distribution_pct": action_distribution,
        },
    ]
)

single_day_summary

## 13. Đánh giá mở rộng trên nhiều ngày

Để có góc nhìn chắc hơn, ta đánh giá trên tất cả ngày hoàn chỉnh trong dataset. Các controller được so sánh:

- `DQN`: chính sách học được.
- `Human`: hành vi người dùng trong dữ liệu.
- `Off_Closed`: luôn tắt điều hòa và đóng cửa.
- `Window_Open`: luôn tắt điều hòa và mở cửa.
- `AC_25_Closed`: luôn bật điều hòa ở 25 độ C và đóng cửa.
- `AC_26_Closed`: luôn bật điều hòa ở 26 độ C và đóng cửa.
- `AC_27_Closed`: luôn bật điều hòa ở 27 độ C và đóng cửa.

Bảng này rất quan trọng vì nó cho thấy DQN học được xu hướng không dùng điều hòa, giúp giảm chi phí năng lượng trong reward nhưng có thể làm comfort thấp hơn human baseline hoặc các baseline bật AC cố định.

In [ ]:
def evaluate_many_days(model_xgb, q_network, data, day_indices, num_actions):
    controllers = {
        "DQN": None,
        "Human": "human",
        "Off_Closed": 0,
        "Window_Open": 12,
        "AC_25_Closed": 6,
        "AC_26_Closed": 7,
        "AC_27_Closed": 8,
    }
    rows_by_controller = {name: [] for name in controllers}
    actions_by_controller = {name: [] for name in controllers}

    for day_index in day_indices:
        data_test_i, xgboost_test_i = choose_day(day_index * 24, data)

        for name, fixed_action in controllers.items():
            if fixed_action == "human":
                rows_by_controller[name].append(summarize_human_with_actions(data_test_i, xgboost_test_i))
                continue

            eval_day_i, actions_i, rewards_i, _ = rollout_day(
                model_xgb,
                q_network,
                data_test_i,
                xgboost_test_i,
                epsilon=0.0,
                train=False,
                train_cfg={"num_actions": num_actions},
                fixed_action=fixed_action,
            )
            metrics = summarize_day(eval_day_i, actions_i)
            metrics["total_reward"] = float(np.sum(rewards_i))
            rows_by_controller[name].append(metrics)
            actions_by_controller[name].extend(int(action) for action in actions_i)

    summary = []
    action_distributions = {}
    for name, rows in rows_by_controller.items():
        metrics = aggregate_metric_rows(rows)
        summary.append({"controller": name, **metrics})

        actions = actions_by_controller.get(name, [])
        if actions:
            counts = pd.Series(actions).value_counts().sort_index()
            action_distributions[name] = {
                int(action): float(count / len(actions) * 100)
                for action, count in counts.items()
            }

    return summary, action_distributions


max_complete_days = max(1, (len(data) - 24) // 24)
eval_indices = list(range(max_complete_days))

evaluation_summary, evaluation_action_distribution = evaluate_many_days(
    model_xgb,
    q_network,
    data,
    eval_indices,
    num_actions,
)

evaluation_df = pd.DataFrame(evaluation_summary)
evaluation_df

## 14. Lưu kết quả để cập nhật báo cáo

Sau khi chạy xong, notebook lưu các artifact sau:

- `training_history.csv`: reward, epsilon và loss trung bình theo episode.
- `summary_metrics.csv`: bảng so sánh nhiều controller trên nhiều ngày.
- `result.json`: toàn bộ kết quả chính, được script tạo DOCX đọc lại để cập nhật báo cáo.
- `whulx_dqn_reproduction.weights.h5`: trọng số DQN sau huấn luyện.

Khi đưa lên Git, nên commit notebook, script/README, dữ liệu CSV và các file kết quả nhỏ như JSON/CSV nếu cần minh họa. Không nên commit file weight `.h5` nếu kích thước lớn.

In [ ]:
result = {
    "episodes": EPISODES,
    "day_index": DAY_INDEX,
    "raw_shape": list(raw.shape),
    "after_clean_shape": list(data.shape),
    "xgboost": xgb_metrics,
    "human_baseline": human_metrics,
    "dqn": {
        **dqn_metrics,
        "total_reward": float(np.sum(eval_rewards)),
        "actions": [int(a) for a in eval_actions],
        "action_distribution_pct": action_distribution,
    },
    "reported_whulx_readme": {
        "comfort_duration_increase_pct": 24.0,
        "ac_usage_decrease_pct": 24.7,
    },
    "evaluation": {
        "eval_days": max_complete_days,
        "controllers": evaluation_summary,
        "action_distribution_pct": evaluation_action_distribution,
    },
}

history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
evaluation_df.to_csv(OUT_DIR / "summary_metrics.csv", index=False)
(OUT_DIR / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
q_network.save_weights(OUT_DIR / "whulx_dqn_reproduction.weights.h5")

print("Đã lưu kết quả vào:", OUT_DIR)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 15. Diễn giải kết quả mới nhất

Theo file `artifacts/outputs/whulx_reproduction/result.json` hiện có trong workspace, pipeline cho kết quả chính như sau:

- Dữ liệu gốc có 42,934 dòng và 27 cột.
- Sau khi làm sạch còn 42,338 dòng và 27 cột.
- XGBoost dùng 42,338 mẫu với 21 biến đầu vào.
- Tập train có 33,870 mẫu, tập test có 8,468 mẫu.
- XGBoost đạt MAE khoảng 0.1823 độ C, RMSE khoảng 0.3482 độ C và R2 khoảng 0.4797.
- Trên ngày mẫu đầu tiên, human baseline đạt 100% comfort và DQN cũng đạt 100% comfort trong khung điều khiển 6-17h.
- Trong ngày mẫu, DQN không bật điều hòa, mở cửa sổ 41.67% số bước điều khiển.
- Khi đánh giá trên 1,763 ngày hoàn chỉnh, DQN đạt 67.99% comfort, AC on 0.00% và window open 34.09%.

Nhận xét quan trọng: pipeline đã tái hiện được logic XGB-DQN, nhưng reward hiện tại phạt chi phí điều hòa mạnh nên chính sách học được thiên về không dùng AC. Vì vậy, kết quả này phù hợp để xác nhận pipeline và làm baseline tái hiện. Nếu mục tiêu tiếp theo là tăng comfort, cần tinh chỉnh lại reward, thêm ràng buộc comfort hoặc mở rộng đánh giá sang môi trường mô phỏng EnergyPlus/Sinergym.